In [5]:
import os
import sys
import yaml
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if os.path.basename(os.getcwd()) != 'audio_event_detection':
    os.chdir('..')
from models.ast_model import AudioSpectrogramTransformer
from scripts.train import Trainer
 
from scripts.evaluate import ModelEvaluator
from utils.dataset import AudioEventDataset

In [6]:
# Load tập test
config_path = 'configs/config.yaml'
metadata_df = pd.read_csv('data/processed/spectrograms/processed_metadata.csv')
train_df, temp_df = train_test_split(metadata_df, test_size=0.2, random_state=42, stratify=metadata_df['label'])

with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)
batch_size = config_dict['training']['batch_size']
num_workers = config_dict.get('hardware', {}).get('num_workers', 4)
pin_memory = config_dict.get('hardware', {}).get('pin_memory', True)

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
test_dataset = AudioEventDataset(test_df, config_path, mode='test')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, 
                         num_workers=num_workers, pin_memory=pin_memory)


In [7]:
best_model_path = os.path.join(os.getcwd(), config_dict['paths']['checkpoint_dir'], 'best_model.pth')
Evaluator = ModelEvaluator(
    model_path=best_model_path
)

Evaluator initialized on cpu


In [8]:
Evaluator.evaluate(test_loader)


Evaluating model...


Evaluation: 100%|██████████| 34/34 [05:04<00:00,  8.96s/it]


{'metrics': {'accuracy': 0.9413407821229051,
  'precision': 0.7985078174647937,
  'recall': 0.6642722154168466,
  'f1_score': 0.7072245401556019,
  'precision_gunshot': 0.918918918918919,
  'recall_gunshot': 0.8947368421052632,
  'f1_gunshot': 0.9066666666666666,
  'precision_explosion': 0.5,
  'recall_explosion': 0.25,
  'f1_explosion': 0.3333333333333333,
  'precision_siren': 0.8924731182795699,
  'recall_siren': 0.8924731182795699,
  'f1_siren': 0.8924731182795699,
  'precision_glass_breaking': 0.5,
  'recall_glass_breaking': 0.25,
  'f1_glass_breaking': 0.3333333333333333,
  'precision_scream': 0.8108108108108109,
  'recall_scream': 0.9,
  'f1_scream': 0.8530805687203792,
  'precision_dog_bark': 1.0,
  'recall_dog_bark': 0.5,
  'f1_dog_bark': 0.6666666666666666,
  'precision_fire_crackling': 0.9673518742442564,
  'recall_fire_crackling': 0.9626955475330926,
  'f1_fire_crackling': 0.9650180940892642,
  'roc_auc': nan,
  'map': 0.6563750866166056},
 'confusion_matrix': array([[ 34,  